# 🏠 House Price Prediction — Complete Linear Regression Walkthrough
## Dataset: House Prices (Kaggle-style) | Author: Sruthi Tarimana

---
### 📋 What this notebook covers:
| # | Topic |
|---|-------|
| 1 | Dataset Overview & Loading |
| 2 | Exploratory Data Analysis (EDA) |
| 3 | Data Cleaning & Preprocessing |
| 4 | Feature Engineering |
| 5 | Assumption Checks (LINNE) |
| 6 | Simple Linear Regression |
| 7 | Multiple Linear Regression |
| 8 | Model Evaluation (R², RMSE, MAE, MAPE) |
| 9 | Residual Diagnostics |
| 10 | Ridge, Lasso & ElasticNet Regularisation |
| 11 | Cross-Validation & Hyperparameter Tuning |
| 12 | Feature Importance |
| 13 | Polynomial Regression |
| 14 | Real Data Prediction |
| 15 | Model Comparison Summary |

> **Dataset:** House Prices dataset (built-in synthetic — mirrors Kaggle House Prices structure)  
> **Target variable:** `SalePrice` (house sale price in USD)


## 📦 Section 1 — Import Libraries

In [ ]:
# ── Core Libraries ────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Sklearn: Preprocessing ─────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split, cross_val_score,
    KFold, GridSearchCV, learning_curve
)
from sklearn.preprocessing import (
    StandardScaler, LabelEncoder, PolynomialFeatures
)
from sklearn.pipeline import Pipeline

# ── Sklearn: Models ─────────────────────────────────────────────────────
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    RidgeCV, LassoCV
)

# ── Sklearn: Metrics ─────────────────────────────────────────────────────
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    r2_score, mean_absolute_percentage_error
)

# ── Stats ─────────────────────────────────────────────────────────────────
from scipy import stats
from scipy.stats import shapiro, jarque_bera

# ── Styling ──────────────────────────────────────────────────────────────
plt.rcParams['figure.dpi']       = 120
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False
sns.set_palette('husl')

print("✅ All libraries imported successfully!")
print(f"   numpy     {np.__version__}")
print(f"   pandas    {pd.__version__}")


## 📂 Section 2 — Dataset Loading
### About the Dataset
- **Source**: Kaggle House Prices — Advanced Regression Techniques (mirrored structure)
- **Rows**: 1,460 houses  
- **Features**: 20 columns covering size, quality, location, and age  
- **Target**: `SalePrice` — the sale price of each house in USD

> 💡 We generate this dataset synthetically with the same statistical properties as the Kaggle version so the notebook runs without downloading files.


In [ ]:
# ── Generate Kaggle-style House Price Dataset ──────────────────────────
np.random.seed(42)
n = 1460

# Core features
GrLivArea    = np.random.normal(1500, 500, n).clip(300, 4000).astype(int)
OverallQual  = np.random.choice(range(1, 11), n,
                   p=[0.01,0.02,0.04,0.07,0.14,0.20,0.22,0.18,0.08,0.04])
YearBuilt    = np.random.randint(1900, 2010, n)
TotalBsmtSF  = (GrLivArea * np.random.uniform(0.3, 1.0, n)).astype(int)
GarageCars   = np.random.choice([0,1,2,3,4], n, p=[0.05,0.15,0.55,0.20,0.05])
FullBath     = np.random.choice([0,1,2,3], n, p=[0.02,0.30,0.55,0.13])
BedroomAbvGr = np.random.choice([1,2,3,4,5], n, p=[0.05,0.18,0.52,0.20,0.05])
LotArea      = np.random.normal(10000, 4000, n).clip(1500, 50000).astype(int)
Neighborhood = np.random.choice(
    ['NAmes','CollgCr','OldTown','Edwards','Somerst','NridgHt','BrkSide'],
    n, p=[0.15,0.15,0.12,0.12,0.11,0.10,0.25])
HouseStyle   = np.random.choice(
    ['1Story','2Story','1.5Fin','SFoyer'], n, p=[0.50,0.30,0.12,0.08])
CentralAir   = np.random.choice(['Y','N'], n, p=[0.93,0.07])
KitchenQual  = np.random.choice(['Ex','Gd','TA','Fa'], n, p=[0.10,0.40,0.42,0.08])
Fireplaces   = np.random.choice([0,1,2,3], n, p=[0.47,0.40,0.11,0.02])
GarageArea   = (GarageCars * np.random.normal(220, 40, n)).clip(0, 1100).astype(int)
MasVnrArea   = np.random.choice(
    [0] + list(np.random.randint(1, 800, 500)), n)
WoodDeckSF   = np.random.choice(
    [0] * 600 + list(np.random.randint(10, 600, 400)), n)
OpenPorchSF  = np.random.choice(
    [0] * 500 + list(np.random.randint(10, 400, 500)), n)
YearRemodAdd = np.where(np.random.rand(n) > 0.4,
                         np.random.randint(1960, 2010, n), YearBuilt)
MSZoning     = np.random.choice(
    ['RL','RM','FV','RH'], n, p=[0.78,0.12,0.07,0.03])

# ── Target: SalePrice (realistic formula + noise) ──────────────────────
noise = np.random.normal(0, 8000, n)
neigh_premium = {'NAmes':0,'CollgCr':5000,'OldTown':-3000,
                 'Edwards':-4000,'Somerst':8000,'NridgHt':20000,'BrkSide':-2000}
neigh_effect  = np.array([neigh_premium[x] for x in Neighborhood])
qual_effect   = np.array([1000,2000,5000,9000,14000,20000,28000,38000,50000,65000])
qual_val      = np.array([qual_effect[q-1] for q in OverallQual])
SalePrice = (
    80000
    + GrLivArea  * 75
    + TotalBsmtSF * 30
    + GarageArea  * 40
    + qual_val
    + neigh_effect
    + (2024 - YearBuilt) * (-200)
    + Fireplaces  * 5000
    + noise
).clip(50000, 750000).astype(int)

# ── Build DataFrame ────────────────────────────────────────────────────
df = pd.DataFrame({
    'Id':           range(1, n+1),
    'MSZoning':     MSZoning,
    'LotArea':      LotArea,
    'Neighborhood': Neighborhood,
    'HouseStyle':   HouseStyle,
    'OverallQual':  OverallQual,
    'YearBuilt':    YearBuilt,
    'YearRemodAdd': YearRemodAdd,
    'MasVnrArea':   MasVnrArea,
    'TotalBsmtSF':  TotalBsmtSF,
    'CentralAir':   CentralAir,
    'GrLivArea':    GrLivArea,
    'FullBath':     FullBath,
    'BedroomAbvGr': BedroomAbvGr,
    'KitchenQual':  KitchenQual,
    'Fireplaces':   Fireplaces,
    'GarageArea':   GarageArea,
    'GarageCars':   GarageCars,
    'WoodDeckSF':   WoodDeckSF,
    'OpenPorchSF':  OpenPorchSF,
    'SalePrice':    SalePrice
})

# Inject 30 missing values for realism
for col in ['MasVnrArea','GarageArea','TotalBsmtSF']:
    idx = np.random.choice(df.index, 10, replace=False)
    df.loc[idx, col] = np.nan

print(f"✅ Dataset created: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()


## 🔍 Section 3 — Exploratory Data Analysis (EDA)

In [ ]:
# ── Basic Info ────────────────────────────────────────────────────────
print("=" * 55)
print("DATASET INFO")
print("=" * 55)
print(f"Shape        : {df.shape}")
print(f"Memory usage : {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
print("\nColumn dtypes:")
print(df.dtypes.value_counts())


In [ ]:
# ── Statistical Summary ───────────────────────────────────────────────
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean','std'])


In [ ]:
# ── Missing Values ────────────────────────────────────────────────────
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
pct     = (missing / len(df) * 100).round(2)
miss_df = pd.DataFrame({'Missing':missing, 'Percent(%)':pct})
print("Missing Values:")
print(miss_df)


In [ ]:
# ── Target Variable Distribution ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Histogram
axes[0].hist(df['SalePrice'], bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_title('SalePrice Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sale Price ($)')
axes[0].set_ylabel('Count')
axes[0].axvline(df['SalePrice'].mean(), color='red', lw=2, linestyle='--', label=f"Mean: ${df['SalePrice'].mean():,.0f}")
axes[0].legend()

# Log-transformed
axes[1].hist(np.log1p(df['SalePrice']), bins=50, color='teal', edgecolor='white', linewidth=0.5)
axes[1].set_title('Log(SalePrice) Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(1 + SalePrice)')
axes[1].set_ylabel('Count')

# Box plot
axes[2].boxplot(df['SalePrice'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='red', linewidth=2))
axes[2].set_title('SalePrice Boxplot', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Sale Price ($)')
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.suptitle('Target Variable Analysis — SalePrice', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Skewness : {df['SalePrice'].skew():.3f}  (>1 = right-skewed, may need log transform)")
print(f"Kurtosis : {df['SalePrice'].kurtosis():.3f}")


In [ ]:
# ── Correlation Heatmap ───────────────────────────────────────────────
num_cols = df.select_dtypes(include=np.number).columns.drop('Id')
corr = df[num_cols].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5,
            annot_kws={'size': 8}, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Top correlated features with target
print("\nTop Features Correlated with SalePrice:")
top_corr = corr['SalePrice'].drop('SalePrice').sort_values(ascending=False)
print(top_corr.to_string())


In [ ]:
# ── Key Feature vs SalePrice Scatter Plots ────────────────────────────
top_features = ['GrLivArea','OverallQual','TotalBsmtSF','GarageArea','YearBuilt','LotArea']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    axes[i].scatter(df[feat], df['SalePrice'], alpha=0.4, s=15, color='steelblue')
    # Trend line
    z = np.polyfit(df[feat].dropna(), df.loc[df[feat].notna(), 'SalePrice'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[feat].min(), df[feat].max(), 200)
    axes[i].plot(x_line, p(x_line), 'r-', lw=2, label='Trend')
    axes[i].set_xlabel(feat, fontsize=10)
    axes[i].set_ylabel('SalePrice ($)', fontsize=10)
    axes[i].set_title(f'{feat} vs SalePrice', fontsize=11, fontweight='bold')
    axes[i].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
    r = df[[feat,'SalePrice']].dropna().corr().iloc[0,1]
    axes[i].text(0.05, 0.92, f'r = {r:.3f}', transform=axes[i].transAxes,
                 fontsize=9, color='darkred', fontweight='bold')

plt.suptitle('Top Features vs SalePrice', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── Categorical Features vs SalePrice ─────────────────────────────────
cat_cols = ['OverallQual','Neighborhood','KitchenQual','CentralAir','GarageCars']

fig, axes = plt.subplots(1, len(cat_cols), figsize=(20, 5))

for i, col in enumerate(cat_cols):
    order = df.groupby(col)['SalePrice'].median().sort_values().index
    sns.boxplot(data=df, x=col, y='SalePrice', order=order,
                palette='husl', ax=axes[i])
    axes[i].set_title(f'{col}\nvs SalePrice', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(col, fontsize=9)
    axes[i].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('Categorical Features vs SalePrice', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 🔧 Section 4 — Data Cleaning & Preprocessing

In [ ]:
# ── 4.1 Handle Missing Values ─────────────────────────────────────────
print("Before imputation:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Numeric: fill with median
for col in ['MasVnrArea','GarageArea','TotalBsmtSF']:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)
    print(f"  ✅ {col}: filled {df[col].isnull().sum()} NaN → median ({median_val:.0f})")

print("\nAfter imputation:", df.isnull().sum().sum(), "missing values")


In [ ]:
# ── 4.2 Detect & Handle Outliers (IQR Method) ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before outlier removal
axes[0].scatter(df['GrLivArea'], df['SalePrice'], alpha=0.4, s=15, color='steelblue')
axes[0].set_title('GrLivArea vs SalePrice\n(Before Outlier Removal)', fontweight='bold')
axes[0].set_xlabel('GrLivArea (sq ft)')
axes[0].set_ylabel('SalePrice ($)')

# Remove outliers: very large houses with very low price
outlier_mask = ~((df['GrLivArea'] > 4000) & (df['SalePrice'] < 200000))
df_clean = df[outlier_mask].copy()
print(f"Removed {len(df) - len(df_clean)} outliers. Remaining: {len(df_clean)} rows")

axes[1].scatter(df_clean['GrLivArea'], df_clean['SalePrice'], alpha=0.4, s=15, color='teal')
axes[1].set_title('GrLivArea vs SalePrice\n(After Outlier Removal)', fontweight='bold')
axes[1].set_xlabel('GrLivArea (sq ft)')
axes[1].set_ylabel('SalePrice ($)')

plt.tight_layout()
plt.show()

df = df_clean.copy()


In [ ]:
# ── 4.3 Encode Categorical Variables ──────────────────────────────────
# Ordinal encoding for quality columns
quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1}
df['KitchenQual_enc'] = df['KitchenQual'].map(quality_map).fillna(3)
df['CentralAir_enc']  = (df['CentralAir'] == 'Y').astype(int)

# One-Hot Encoding for Neighborhood and HouseStyle
df = pd.get_dummies(df, columns=['Neighborhood','HouseStyle','MSZoning'], drop_first=True)

# Label encode remaining categoricals
cat_remaining = df.select_dtypes(include='object').columns
print("Remaining categoricals:", list(cat_remaining))
for col in cat_remaining:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

print(f"\n✅ Encoding done. Dataset shape: {df.shape}")


In [ ]:
# ── 4.4 Feature Engineering ───────────────────────────────────────────
# Age of house at time of sale
df['HouseAge']      = 2010 - df['YearBuilt']
df['YearsSinceRemod'] = 2010 - df['YearRemodAdd']
df['TotalSF']       = df['TotalBsmtSF'] + df['GrLivArea']
df['TotalBath']     = df['FullBath'] + 0.5 * df.get('HalfBath', 0)
df['HasGarage']     = (df['GarageArea'] > 0).astype(int)
df['HasFireplace']  = (df['Fireplaces'] > 0).astype(int)
df['HasDeck']       = (df['WoodDeckSF'] > 0).astype(int)
df['QualArea']      = df['OverallQual'] * df['GrLivArea']   # interaction feature

print("✅ New features created:")
new_feats = ['HouseAge','YearsSinceRemod','TotalSF','TotalBath',
             'HasGarage','HasFireplace','HasDeck','QualArea']
print(df[new_feats].describe().T[['mean','std','min','max']].round(2))


## 📐 Section 5 — Assumption Checks (LINNE)

Before fitting Linear Regression, we check all **5 assumptions**:
1. **L**inearity
2. **I**ndependence  
3. **N**ormality of errors
4. **N**o Multicollinearity
5. **E**qual variance (Homoscedasticity)


In [ ]:
# ── 5.1 Linearity Check ───────────────────────────────────────────────
key_feats = ['GrLivArea','TotalSF','OverallQual','GarageArea']
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for i, feat in enumerate(key_feats):
    axes[i].scatter(df[feat], df['SalePrice'], alpha=0.3, s=10, color='steelblue')
    z = np.polyfit(df[feat], df['SalePrice'], 1)
    xr = np.linspace(df[feat].min(), df[feat].max(), 200)
    axes[i].plot(xr, np.poly1d(z)(xr), 'r-', lw=2)
    r = np.corrcoef(df[feat], df['SalePrice'])[0,1]
    axes[i].set_title(f'{feat}\nr = {r:.3f}', fontweight='bold', fontsize=10)
    axes[i].set_xlabel(feat); axes[i].set_ylabel('SalePrice')

plt.suptitle('Linearity Check — Feature vs Target', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("✅ Linearity: GrLivArea, TotalSF, OverallQual show strong linear relationship.")


In [ ]:
# ── 5.2 Multicollinearity Check (VIF) ────────────────────────────────
try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    num_df = df[['GrLivArea','TotalBsmtSF','GarageArea','OverallQual',
                 'HouseAge','LotArea','TotalSF','QualArea']].dropna()
    vif_df = pd.DataFrame({
        'Feature': num_df.columns,
        'VIF': [variance_inflation_factor(num_df.values, i)
                for i in range(num_df.shape[1])]
    }).sort_values('VIF', ascending=False)
    print("VIF Results:")
    print(vif_df.to_string(index=False))
    print("\n  VIF < 5  : OK | VIF 5-10 : Moderate | VIF > 10 : Problem")
except ImportError:
    # If statsmodels not available, use correlation proxy
    num_df = df[['GrLivArea','TotalBsmtSF','GarageArea','OverallQual',
                 'HouseAge','LotArea','TotalSF']].dropna()
    corr_matrix = num_df.corr().abs()
    print("Correlation Matrix (proxy for multicollinearity check):")
    print(corr_matrix.round(2))
    high_corr = [(corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i,j])
                 for i in range(len(corr_matrix))
                 for j in range(i+1, len(corr_matrix))
                 if corr_matrix.iloc[i,j] > 0.8]
    if high_corr:
        print("\n⚠️  Highly correlated pairs (>0.8):")
        for a,b,v in high_corr:
            print(f"   {a} <-> {b}  : {v:.3f}")
    else:
        print("\n✅ No severe multicollinearity detected.")


## ✂️ Section 6 — Train / Test Split & Feature Scaling

In [ ]:
# ── Select Features ──────────────────────────────────────────────────
feature_cols = [
    'GrLivArea','TotalBsmtSF','GarageArea','OverallQual',
    'HouseAge','LotArea','Fireplaces','FullBath','BedroomAbvGr',
    'KitchenQual_enc','CentralAir_enc','HasFireplace','HasDeck',
    'TotalSF','QualArea','YearsSinceRemod'
]
# Keep only columns that exist after encoding
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols].copy()
y = df['SalePrice'].copy()

print(f"Features selected: {len(feature_cols)}")
print(f"X shape: {X.shape}  |  y shape: {y.shape}")
print(f"\nFeatures used:")
for i, f in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {f}")


In [ ]:
# ── Train / Test Split ────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train set : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test  set : {X_test.shape[0]}  samples ({X_test.shape[0]/len(X)*100:.0f}%)")

# ── Feature Scaling ────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit ONLY on train
X_test_sc  = scaler.transform(X_test)         # transform test

print("\n✅ StandardScaler applied (fit on train, transform on test)")
print(f"   Train mean (post-scale): {X_train_sc.mean():.6f}  std: {X_train_sc.std():.4f}")


## 📈 Section 7 — Simple Linear Regression (GrLivArea → SalePrice)

In [ ]:
# ── Simple Linear Regression ──────────────────────────────────────────
X_simple_train = X_train[['GrLivArea']]
X_simple_test  = X_test[['GrLivArea']]

slr = LinearRegression()
slr.fit(X_simple_train, y_train)
y_pred_slr = slr.predict(X_simple_test)

print("═" * 45)
print("SIMPLE LINEAR REGRESSION RESULTS")
print("═" * 45)
print(f"  Intercept (b0) : ${slr.intercept_:,.2f}")
print(f"  Slope     (b1) : {slr.coef_[0]:.4f}  (per sq ft)")
print(f"  Equation       : SalePrice = {slr.intercept_:,.0f} + {slr.coef_[0]:.2f} × GrLivArea")
print()
print(f"  R²   (test)    : {r2_score(y_test, y_pred_slr):.4f}")
print(f"  RMSE (test)    : ${np.sqrt(mean_squared_error(y_test, y_pred_slr)):,.2f}")
print(f"  MAE  (test)    : ${mean_absolute_error(y_test, y_pred_slr):,.2f}")
print("═" * 45)


In [ ]:
# ── Visualise Simple LR ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter + regression line
x_range = np.linspace(X_test['GrLivArea'].min(), X_test['GrLivArea'].max(), 300).reshape(-1,1)
y_range  = slr.predict(x_range)

axes[0].scatter(X_test['GrLivArea'], y_test, alpha=0.4, s=20, color='steelblue', label='Actual')
axes[0].plot(x_range, y_range, 'r-', lw=2.5, label='Regression Line')
axes[0].set_xlabel('GrLivArea (sq ft)', fontsize=11)
axes[0].set_ylabel('Sale Price ($)', fontsize=11)
axes[0].set_title('Simple LR: GrLivArea → SalePrice', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))

# Actual vs Predicted
axes[1].scatter(y_test, y_pred_slr, alpha=0.4, s=20, color='teal')
mn, mx = y_test.min(), y_test.max()
axes[1].plot([mn,mx],[mn,mx], 'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual SalePrice', fontsize=11)
axes[1].set_ylabel('Predicted SalePrice', fontsize=11)
axes[1].set_title('Actual vs Predicted (Simple LR)', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.show()


## 📊 Section 8 — Multiple Linear Regression

In [ ]:
# ── Multiple Linear Regression ────────────────────────────────────────
mlr = LinearRegression()
mlr.fit(X_train_sc, y_train)
y_pred_mlr_train = mlr.predict(X_train_sc)
y_pred_mlr_test  = mlr.predict(X_test_sc)

print("═" * 55)
print("MULTIPLE LINEAR REGRESSION RESULTS")
print("═" * 55)
print(f"  Intercept (b0) : ${mlr.intercept_:,.2f}")
print()
print("  Feature Coefficients:")
coef_df = pd.DataFrame({
    'Feature'    : feature_cols,
    'Coefficient': mlr.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
for _, row in coef_df.iterrows():
    bar = '▮' * int(abs(row['Coefficient']) / 5000)
    sign = '+' if row['Coefficient'] > 0 else '-'
    print(f"    {row['Feature']:<22} {sign}${abs(row['Coefficient']):>10,.0f}  {bar}")
print()
print(f"  Train R²  : {r2_score(y_train, y_pred_mlr_train):.4f}")
print(f"  Test  R²  : {r2_score(y_test, y_pred_mlr_test):.4f}")
print(f"  Train RMSE: ${np.sqrt(mean_squared_error(y_train, y_pred_mlr_train)):,.2f}")
print(f"  Test  RMSE: ${np.sqrt(mean_squared_error(y_test, y_pred_mlr_test)):,.2f}")
print(f"  Test  MAE : ${mean_absolute_error(y_test, y_pred_mlr_test):,.2f}")
print(f"  Test  MAPE: {mean_absolute_percentage_error(y_test, y_pred_mlr_test)*100:.2f}%")
print("═" * 55)


In [ ]:
# ── Visualise MLR Results ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred_mlr_test, alpha=0.4, s=20, color='darkorange')
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn,mx],[mn,mx], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual SalePrice', fontsize=11)
axes[0].set_ylabel('Predicted SalePrice', fontsize=11)
r2 = r2_score(y_test, y_pred_mlr_test)
axes[0].set_title(f'Actual vs Predicted\nMultiple LR  |  R² = {r2:.4f}', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))

# Coefficient plot
coef_plot = coef_df.head(12)
colors = ['steelblue' if v > 0 else 'salmon' for v in coef_plot['Coefficient']]
bars = axes[1].barh(coef_plot['Feature'], coef_plot['Coefficient'], color=colors)
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_xlabel('Coefficient Value (scaled)', fontsize=11)
axes[1].set_title('Feature Coefficients\n(Top 12 by magnitude)', fontsize=12, fontweight='bold')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.show()


## 📏 Section 9 — Complete Evaluation Metrics

In [ ]:
# ── All Metrics in One Place ──────────────────────────────────────────
def evaluate_model(name, y_true, y_pred, y_train_true=None, y_train_pred=None):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    n, k = len(y_true), X_train.shape[1]
    adj_r2 = 1 - (1-r2)*(n-1)/(n-k-1)

    print(f"\n{'─'*50}")
    print(f"  MODEL: {name}")
    print(f"{'─'*50}")
    print(f"  R²           : {r2:.4f}")
    print(f"  Adj R²       : {adj_r2:.4f}")
    print(f"  RMSE         : ${rmse:,.2f}")
    print(f"  MAE          : ${mae:,.2f}")
    print(f"  MAPE         : {mape:.2f}%")
    print(f"  MSE          : {mse:,.0f}")
    if y_train_true is not None and y_train_pred is not None:
        r2_train = r2_score(y_train_true, y_train_pred)
        rmse_tr  = np.sqrt(mean_squared_error(y_train_true, y_train_pred))
        print(f"  Train R²     : {r2_train:.4f}  (gap = {r2_train - r2:.4f})")
        print(f"  Train RMSE   : ${rmse_tr:,.2f}")
        if r2_train - r2 > 0.10:
            print("  ⚠️  Large train-test gap detected → possible OVERFITTING")
        else:
            print("  ✅  Train-test gap acceptable → good generalisation")
    return {'name':name,'R2':r2,'AdjR2':adj_r2,'RMSE':rmse,'MAE':mae,'MAPE':mape}

results = []
results.append(evaluate_model(
    "Simple LR (GrLivArea only)",
    y_test, y_pred_slr
))
results.append(evaluate_model(
    "Multiple LR (all features)",
    y_test, y_pred_mlr_test,
    y_train, y_pred_mlr_train
))


## 🔬 Section 10 — Residual Diagnostics (Assumption Verification)

In [ ]:
# ── Residual Analysis — Complete Diagnostic Plots ────────────────────
residuals = y_test - y_pred_mlr_test
std_resid  = (residuals - residuals.mean()) / residuals.std()

fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# 1. Residuals vs Fitted
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(y_pred_mlr_test, residuals, alpha=0.4, s=15, color='steelblue')
ax1.axhline(0, color='red', lw=2, linestyle='--')
ax1.set_xlabel('Fitted Values', fontsize=10)
ax1.set_ylabel('Residuals', fontsize=10)
ax1.set_title('Residuals vs Fitted\n(checks linearity & homoscedasticity)',
              fontsize=10, fontweight='bold')
z = np.polyfit(y_pred_mlr_test, residuals, 2)
xr = np.linspace(y_pred_mlr_test.min(), y_pred_mlr_test.max(), 200)
ax1.plot(xr, np.poly1d(z)(xr), 'r-', lw=1.5, alpha=0.6)

# 2. Q-Q Plot (Normality)
ax2 = fig.add_subplot(gs[0, 1])
stats.probplot(residuals, dist="norm", plot=ax2)
ax2.set_title('Q-Q Plot of Residuals\n(checks normality of errors)',
              fontsize=10, fontweight='bold')
ax2.get_lines()[0].set(markersize=3, alpha=0.5, color='steelblue')
ax2.get_lines()[1].set(color='red', lw=2)

# 3. Residual Histogram
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(residuals, bins=40, color='teal', edgecolor='white', linewidth=0.4,
         density=True, alpha=0.8)
xr2 = np.linspace(residuals.min(), residuals.max(), 200)
ax3.plot(xr2, stats.norm.pdf(xr2, residuals.mean(), residuals.std()),
         'r-', lw=2, label='Normal curve')
ax3.set_xlabel('Residuals', fontsize=10)
ax3.set_ylabel('Density', fontsize=10)
ax3.set_title('Residual Distribution\n(should be bell-shaped)',
              fontsize=10, fontweight='bold')
ax3.legend()

# 4. Scale-Location (homoscedasticity)
ax4 = fig.add_subplot(gs[1, 0])
ax4.scatter(y_pred_mlr_test, np.sqrt(np.abs(std_resid)),
            alpha=0.4, s=15, color='darkorange')
ax4.set_xlabel('Fitted Values', fontsize=10)
ax4.set_ylabel('√|Standardised Residuals|', fontsize=10)
ax4.set_title('Scale-Location Plot\n(should be flat — checks homoscedasticity)',
              fontsize=10, fontweight='bold')

# 5. Residuals vs Index (independence)
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(np.arange(len(residuals)), residuals, 'o', alpha=0.3, ms=3, color='purple')
ax5.axhline(0, color='red', lw=1.5, linestyle='--')
ax5.set_xlabel('Observation Index', fontsize=10)
ax5.set_ylabel('Residuals', fontsize=10)
ax5.set_title('Residuals vs Index\n(checks independence)',
              fontsize=10, fontweight='bold')

# 6. Residuals distribution stats
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
stat_sw, p_sw = shapiro(residuals[:500])  # Shapiro-Wilk (max 5000)
stat_jb, p_jb, _, _ = jarque_bera(residuals)
table_data = [
    ['Test', 'Statistic', 'p-value', 'Pass?'],
    ['Shapiro-Wilk\n(Normality)', f'{stat_sw:.4f}', f'{p_sw:.4f}', '✅' if p_sw > 0.05 else '⚠️'],
    ['Jarque-Bera\n(Normality)', f'{stat_jb:.4f}', f'{p_jb:.4f}', '✅' if p_jb > 0.05 else '⚠️'],
    ['Mean of residuals', f'{residuals.mean():.2f}', '≈0?', '✅' if abs(residuals.mean()) < 1000 else '⚠️'],
    ['Std of residuals', f'{residuals.std():,.0f}', 'RMSE', '—'],
]
tbl = ax6.table(cellText=table_data[1:], colLabels=table_data[0],
                cellLoc='center', loc='center', bbox=[0,0,1,1])
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
for (r,c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#f0f4f8')
ax6.set_title('Normality Test Results', fontsize=10, fontweight='bold', pad=10)

plt.suptitle('Complete Residual Diagnostic Plots', fontsize=14, fontweight='bold', y=1.01)
plt.show()


## 🎛️ Section 11 — Ridge, Lasso & ElasticNet Regularisation

In [ ]:
# ── Ridge Regression ─────────────────────────────────────────────────
alphas = [0.001, 0.01, 0.1, 1, 10, 50, 100, 500, 1000]
ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_train_sc, y_train)

print(f"Best Ridge alpha : {ridge_cv.alpha_}")
y_pred_ridge = ridge_cv.predict(X_test_sc)
results.append(evaluate_model(
    f"Ridge (alpha={ridge_cv.alpha_})",
    y_test, y_pred_ridge,
    y_train, ridge_cv.predict(X_train_sc)
))


In [ ]:
# ── Lasso Regression ─────────────────────────────────────────────────
lasso_cv = LassoCV(alphas=np.logspace(-3, 4, 50), cv=5, random_state=42, max_iter=5000)
lasso_cv.fit(X_train_sc, y_train)

print(f"Best Lasso alpha  : {lasso_cv.alpha_:.4f}")
y_pred_lasso = lasso_cv.predict(X_test_sc)
non_zero = np.sum(lasso_cv.coef_ != 0)
print(f"Non-zero coefs    : {non_zero} / {len(feature_cols)} features selected")
print(f"Zeroed-out coefs  : {len(feature_cols) - non_zero} features removed by Lasso")
results.append(evaluate_model(
    f"Lasso (alpha={lasso_cv.alpha_:.4f})",
    y_test, y_pred_lasso,
    y_train, lasso_cv.predict(X_train_sc)
))


In [ ]:
# ── ElasticNet ────────────────────────────────────────────────────────
en_params = {
    'model__alpha':    [0.001, 0.01, 0.1, 1, 10],
    'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}
en_pipe = Pipeline([('model', ElasticNet(max_iter=5000, random_state=42))])
en_grid = GridSearchCV(en_pipe, en_params, cv=5, scoring='r2', n_jobs=-1)
en_grid.fit(X_train_sc, y_train)

print(f"Best ElasticNet params: {en_grid.best_params_}")
print(f"Best CV R²           : {en_grid.best_score_:.4f}")
y_pred_en = en_grid.predict(X_test_sc)
results.append(evaluate_model(
    "ElasticNet (tuned)",
    y_test, y_pred_en,
    y_train, en_grid.predict(X_train_sc)
))


In [ ]:
# ── Regularisation Path — Lasso ───────────────────────────────────────
alphas_path = np.logspace(-2, 5, 100)
coefs_path  = []
for a in alphas_path:
    from sklearn.linear_model import Lasso as L
    lm = L(alpha=a, max_iter=5000)
    lm.fit(X_train_sc, y_train)
    coefs_path.append(lm.coef_)

coefs_path = np.array(coefs_path)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Lasso path
for j in range(coefs_path.shape[1]):
    axes[0].semilogx(alphas_path, coefs_path[:, j], lw=1.2)
axes[0].axvline(lasso_cv.alpha_, color='red', lw=2, linestyle='--',
                label=f'Best alpha = {lasso_cv.alpha_:.4f}')
axes[0].set_xlabel('Alpha (regularisation strength)', fontsize=11)
axes[0].set_ylabel('Coefficient Value', fontsize=11)
axes[0].set_title('Lasso Regularisation Path\n(coefficients shrink to zero as alpha increases)',
                   fontsize=11, fontweight='bold')
axes[0].legend()

# Ridge vs Lasso vs ElasticNet comparison (coefficient magnitudes)
models_coef = {
    'Linear': mlr.coef_,
    'Ridge':  ridge_cv.coef_,
    'Lasso':  lasso_cv.coef_,
}
x_pos = np.arange(len(feature_cols))
width = 0.28
for i, (name_m, coef) in enumerate(models_coef.items()):
    axes[1].bar(x_pos + i*width, np.abs(coef), width, label=name_m, alpha=0.8)
axes[1].set_xticks(x_pos + width)
axes[1].set_xticklabels(feature_cols, rotation=55, ha='right', fontsize=7.5)
axes[1].set_ylabel('|Coefficient| (scaled)', fontsize=11)
axes[1].set_title('Coefficient Comparison\nLinear vs Ridge vs Lasso',
                   fontsize=11, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()


## 🔄 Section 12 — Cross-Validation

In [ ]:
# ── K-Fold Cross-Validation for all models ────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    'Linear Regression': LinearRegression(),
    'Ridge':             Ridge(alpha=ridge_cv.alpha_),
    'Lasso':             Lasso(alpha=lasso_cv.alpha_, max_iter=5000),
    'ElasticNet':        ElasticNet(**{k.replace('model__',''):v
                                       for k,v in en_grid.best_params_.items()},
                                   max_iter=5000),
}

print("5-Fold Cross-Validation Results:")
print(f"{'Model':<25} {'Mean R²':>10} {'Std R²':>10} {'Mean RMSE':>14}")
print("─" * 65)

cv_results = {}
for name_m, model in cv_models.items():
    r2_scores   = cross_val_score(model, X_train_sc, y_train, cv=kf, scoring='r2')
    rmse_scores = np.sqrt(-cross_val_score(model, X_train_sc, y_train,
                                            cv=kf, scoring='neg_mean_squared_error'))
    cv_results[name_m] = {'r2': r2_scores, 'rmse': rmse_scores}
    print(f"{name_m:<25} {r2_scores.mean():>10.4f} {r2_scores.std():>10.4f} "
          f"${rmse_scores.mean():>12,.0f}")

print("─" * 65)


In [ ]:
# ── CV Visualisation ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names_cv = list(cv_results.keys())
r2_means  = [cv_results[n]['r2'].mean()   for n in names_cv]
r2_stds   = [cv_results[n]['r2'].std()    for n in names_cv]
rmse_means= [cv_results[n]['rmse'].mean() for n in names_cv]

colors_cv = ['#2196F3','#4CAF50','#FF9800','#9C27B0']

bars1 = axes[0].bar(names_cv, r2_means, color=colors_cv, alpha=0.85,
                    yerr=r2_stds, capsize=5)
axes[0].set_ylim(0.7, 1.0)
axes[0].set_ylabel('R² Score', fontsize=11)
axes[0].set_title('Cross-Validated R² Scores\n(5-Fold, mean ± std)', fontsize=11, fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)
for bar, val in zip(bars1, r2_means):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

bars2 = axes[1].bar(names_cv, rmse_means, color=colors_cv, alpha=0.85)
axes[1].set_ylabel('RMSE ($)', fontsize=11)
axes[1].set_title('Cross-Validated RMSE\n(5-Fold, mean)', fontsize=11, fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
for bar, val in zip(bars2, rmse_means):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'${val/1000:.1f}K', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


## 📐 Section 13 — Polynomial Regression

In [ ]:
# ── Polynomial Regression (degree 1, 2, 3) ───────────────────────────
X_poly_train = X_train[['GrLivArea']].values
X_poly_test  = X_test[['GrLivArea']].values

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
degrees = [1, 2, 3]

for i, deg in enumerate(degrees):
    poly = PolynomialFeatures(degree=deg)
    X_p_tr = poly.fit_transform(X_poly_train)
    X_p_te = poly.transform(X_poly_test)

    scaler_p = StandardScaler()
    X_p_tr_sc = scaler_p.fit_transform(X_p_tr)
    X_p_te_sc = scaler_p.transform(X_p_te)

    pm = LinearRegression()
    pm.fit(X_p_tr_sc, y_train)
    y_p_pred = pm.predict(X_p_te_sc)

    r2_p   = r2_score(y_test, y_p_pred)
    rmse_p = np.sqrt(mean_squared_error(y_test, y_p_pred))

    x_range = np.linspace(X_poly_train.min(), X_poly_train.max(), 300).reshape(-1,1)
    X_r_p   = scaler_p.transform(poly.transform(x_range))
    y_r_p   = pm.predict(X_r_p)

    axes[i].scatter(X_poly_test, y_test, alpha=0.3, s=10, color='steelblue', label='Actual')
    axes[i].plot(x_range, y_r_p, 'r-', lw=2.5, label='Poly fit')
    axes[i].set_xlabel('GrLivArea (sq ft)', fontsize=10)
    axes[i].set_ylabel('SalePrice ($)', fontsize=10)
    axes[i].set_title(f'Degree {deg} Polynomial\nR²={r2_p:.4f} | RMSE=${rmse_p/1000:.1f}K',
                       fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=8)
    axes[i].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))

    if deg == 1:
        results.append({'name':f'Poly d={deg}','R2':r2_p,'RMSE':rmse_p,'MAE':0,'MAPE':0,'AdjR2':0})

plt.suptitle('Polynomial Regression — Degrees 1, 2, 3', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 🎯 Section 14 — Feature Importance

In [ ]:
# ── Feature Importance via Coefficients & Permutation ────────────────
# Absolute standardised coefficients = relative importance
mlr_final = LinearRegression()
mlr_final.fit(X_train_sc, y_train)

feat_imp = pd.DataFrame({
    'Feature':    feature_cols,
    'Coef':       mlr_final.coef_,
    'AbsCoef':    np.abs(mlr_final.coef_),
    'LassoCoef':  lasso_cv.coef_,
}).sort_values('AbsCoef', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Linear model importance (absolute coefficients)
colors_imp = ['steelblue' if v > 0 else 'salmon' for v in feat_imp['Coef']]
bars = axes[0].barh(feat_imp['Feature'], feat_imp['Coef'], color=colors_imp, alpha=0.85)
axes[0].axvline(0, color='black', lw=0.8)
axes[0].set_xlabel('Coefficient (standardised scale)', fontsize=11)
axes[0].set_title('Linear Regression\nFeature Importance (coefficients)',
                   fontsize=12, fontweight='bold')

# Lasso importance (zero = removed)
feat_imp_lasso = feat_imp.sort_values('LassoCoef', key=abs, ascending=True)
colors_lasso = ['teal' if v != 0 else 'lightgray' for v in feat_imp_lasso['LassoCoef']]
axes[1].barh(feat_imp_lasso['Feature'], feat_imp_lasso['LassoCoef'],
             color=colors_lasso, alpha=0.85)
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_xlabel('Coefficient (Lasso — zero = removed)', fontsize=11)
axes[1].set_title('Lasso Regression\nFeature Selection (non-zero = selected)',
                   fontsize=12, fontweight='bold')

for ax in axes:
    for label in ax.get_yticklabels():
        label.set_fontsize(9)

plt.suptitle('Feature Importance Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\nTop 5 most impactful features (by abs coefficient):")
print(feat_imp.sort_values('AbsCoef', ascending=False).head(5)[['Feature','Coef','LassoCoef']].to_string(index=False))


## 📉 Section 15 — Learning Curve (Bias-Variance Diagnosis)

In [ ]:
# ── Learning Curve ────────────────────────────────────────────────────
train_sizes, train_scores, val_scores = learning_curve(
    LinearRegression(), X_train_sc, y_train,
    cv=5, scoring='r2',
    train_sizes=np.linspace(0.1, 1.0, 15),
    random_state=42
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_sizes, train_mean, 'o-', color='steelblue', lw=2, label='Training R²')
ax.fill_between(train_sizes, train_mean-train_std, train_mean+train_std,
                alpha=0.15, color='steelblue')
ax.plot(train_sizes, val_mean, 'o-', color='darkorange', lw=2, label='Validation R²')
ax.fill_between(train_sizes, val_mean-val_std, val_mean+val_std,
                alpha=0.15, color='darkorange')
ax.axhline(val_mean[-1], color='green', lw=1.5, linestyle=':', alpha=0.7,
           label=f'Final Val R² = {val_mean[-1]:.4f}')
ax.set_xlabel('Training Set Size', fontsize=12)
ax.set_ylabel('R² Score', fontsize=12)
ax.set_title('Learning Curve — Linear Regression\n(Bias-Variance Diagnosis)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0.5, 1.05)
ax.grid(True, alpha=0.3)

gap = train_mean[-1] - val_mean[-1]
if gap > 0.10:
    ax.text(0.6, 0.6, f'⚠️ Gap = {gap:.3f}\nOverfitting!', transform=ax.transAxes,
            fontsize=11, color='red', fontweight='bold')
else:
    ax.text(0.6, 0.6, f'✅ Gap = {gap:.3f}\nGood fit', transform=ax.transAxes,
            fontsize=11, color='green', fontweight='bold')

plt.tight_layout()
plt.show()


## 🏡 Section 16 — Real Data Prediction on New Houses

In [ ]:
# ── Predict Sale Price for New Houses ────────────────────────────────
# Define 5 real-world house profiles to predict
new_houses = pd.DataFrame({
    'GrLivArea'      : [1200,  2500,  800,  3200,  1800],
    'TotalBsmtSF'    : [900,   2000,  600,  2500,  1400],
    'GarageArea'     : [400,   800,   0,    900,   500],
    'OverallQual'    : [5,     8,     4,    9,     6],
    'HouseAge'       : [30,    5,     60,   2,     20],
    'LotArea'        : [8000,  12000, 6000, 15000, 9000],
    'Fireplaces'     : [0,     2,     0,    3,     1],
    'FullBath'       : [1,     3,     1,    4,     2],
    'BedroomAbvGr'   : [3,     4,     2,    5,     3],
    'KitchenQual_enc': [3,     5,     2,    5,     4],
    'CentralAir_enc' : [1,     1,     0,    1,     1],
    'HasFireplace'   : [0,     1,     0,    1,     1],
    'HasDeck'        : [0,     1,     0,    1,     0],
    'TotalSF'        : [2100,  4500,  1400, 5700,  3200],
    'QualArea'       : [6000,  20000, 3200, 28800, 10800],
    'YearsSinceRemod': [30,    3,     60,   1,     15],
})

# Ensure all columns match training features
for col in feature_cols:
    if col not in new_houses.columns:
        new_houses[col] = 0
new_houses = new_houses[feature_cols]

# Scale and predict
new_houses_sc   = scaler.transform(new_houses)
preds_lr        = mlr_final.predict(new_houses_sc)
preds_ridge     = ridge_cv.predict(new_houses_sc)
preds_lasso_new = lasso_cv.predict(new_houses_sc)

house_labels = [
    'Starter Home\n(1200sqft, Qual=5)',
    'Modern Family\n(2500sqft, Qual=8)',
    'Old Small\n(800sqft, Qual=4)',
    'Luxury Estate\n(3200sqft, Qual=9)',
    'Mid-Range\n(1800sqft, Qual=6)',
]

print("=" * 75)
print(f"{'House Profile':<30} {'Linear LR':>14} {'Ridge':>14} {'Lasso':>14}")
print("=" * 75)
for i in range(5):
    print(f"{house_labels[i].replace(chr(10),' '):<30} "
          f"${preds_lr[i]:>12,.0f}  ${preds_ridge[i]:>12,.0f}  ${preds_lasso_new[i]:>12,.0f}")
print("=" * 75)


In [ ]:
# ── Prediction Visualisation ──────────────────────────────────────────
x_idx    = np.arange(5)
width_p  = 0.28
short_labels = ['Starter\n$LQ$','Modern\nFamily','Old\nSmall','Luxury\nEstate','Mid\nRange']

fig, ax = plt.subplots(figsize=(13, 6))
bars_lr    = ax.bar(x_idx - width_p, preds_lr,         width_p, label='Linear LR', color='steelblue', alpha=0.85)
bars_ridge = ax.bar(x_idx,           preds_ridge,       width_p, label='Ridge',     color='teal',      alpha=0.85)
bars_lasso = ax.bar(x_idx + width_p, preds_lasso_new,  width_p, label='Lasso',     color='darkorange', alpha=0.85)

ax.set_xticks(x_idx)
ax.set_xticklabels(house_labels, fontsize=9)
ax.set_ylabel('Predicted Sale Price ($)', fontsize=12)
ax.set_title('Predicted House Prices — New Data\nLinear LR vs Ridge vs Lasso',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
ax.set_ylim(0, max(preds_lr.max(), preds_ridge.max(), preds_lasso_new.max()) * 1.15)

for bars in [bars_lr, bars_ridge, bars_lasso]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
                f'${bar.get_height()/1000:.0f}K',
                ha='center', va='bottom', fontsize=7.5, fontweight='bold')

plt.tight_layout()
plt.show()


## 🏆 Section 17 — Final Model Comparison Summary

In [ ]:
# ── Complete Model Comparison Table ──────────────────────────────────
comparison_data = {
    'Model'     : ['Simple LR', 'Multiple LR', 'Ridge', 'Lasso', 'ElasticNet'],
    'Test R²'   : [
        r2_score(y_test, y_pred_slr),
        r2_score(y_test, y_pred_mlr_test),
        r2_score(y_test, y_pred_ridge),
        r2_score(y_test, y_pred_lasso),
        r2_score(y_test, y_pred_en),
    ],
    'Test RMSE ($)': [
        np.sqrt(mean_squared_error(y_test, y_pred_slr)),
        np.sqrt(mean_squared_error(y_test, y_pred_mlr_test)),
        np.sqrt(mean_squared_error(y_test, y_pred_ridge)),
        np.sqrt(mean_squared_error(y_test, y_pred_lasso)),
        np.sqrt(mean_squared_error(y_test, y_pred_en)),
    ],
    'Test MAE ($)': [
        mean_absolute_error(y_test, y_pred_slr),
        mean_absolute_error(y_test, y_pred_mlr_test),
        mean_absolute_error(y_test, y_pred_ridge),
        mean_absolute_error(y_test, y_pred_lasso),
        mean_absolute_error(y_test, y_pred_en),
    ],
    'MAPE (%)': [
        mean_absolute_percentage_error(y_test, y_pred_slr)*100,
        mean_absolute_percentage_error(y_test, y_pred_mlr_test)*100,
        mean_absolute_percentage_error(y_test, y_pred_ridge)*100,
        mean_absolute_percentage_error(y_test, y_pred_lasso)*100,
        mean_absolute_percentage_error(y_test, y_pred_en)*100,
    ],
}
cmp_df = pd.DataFrame(comparison_data)
cmp_df['Test RMSE ($)'] = cmp_df['Test RMSE ($)'].map(lambda x: f'${x:,.0f}')
cmp_df['Test MAE ($)']  = cmp_df['Test MAE ($)'].map(lambda x: f'${x:,.0f}')
cmp_df['MAPE (%)']      = cmp_df['MAPE (%)'].map(lambda x: f'{x:.2f}%')
cmp_df['Test R²']       = cmp_df['Test R²'].map(lambda x: f'{x:.4f}')

print("\n" + "=" * 65)
print("  FINAL MODEL COMPARISON SUMMARY")
print("=" * 65)
print(cmp_df.to_string(index=False))
print("=" * 65)


In [ ]:
# ── Comparison Bar Chart ─────────────────────────────────────────────
model_names = ['Simple LR','Multiple LR','Ridge','Lasso','ElasticNet']
r2_vals     = [
    r2_score(y_test, y_pred_slr),
    r2_score(y_test, y_pred_mlr_test),
    r2_score(y_test, y_pred_ridge),
    r2_score(y_test, y_pred_lasso),
    r2_score(y_test, y_pred_en),
]
rmse_vals   = [
    np.sqrt(mean_squared_error(y_test, y_pred_slr)),
    np.sqrt(mean_squared_error(y_test, y_pred_mlr_test)),
    np.sqrt(mean_squared_error(y_test, y_pred_ridge)),
    np.sqrt(mean_squared_error(y_test, y_pred_lasso)),
    np.sqrt(mean_squared_error(y_test, y_pred_en)),
]

best_r2  = np.argmax(r2_vals)
best_rmse= np.argmin(rmse_vals)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
palette_cmp = ['#90CAF9','#1E88E5','#4CAF50','#FF9800','#9C27B0']

# R² comparison
bars_r2 = axes[0].bar(model_names, r2_vals, color=palette_cmp, alpha=0.88)
bars_r2[best_r2].set_edgecolor('gold'); bars_r2[best_r2].set_linewidth(3)
axes[0].set_ylim(0.5, 1.05)
axes[0].set_ylabel('R² Score', fontsize=12)
axes[0].set_title('Test R² — All Models\n(higher is better)', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars_r2, r2_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].text(best_r2, r2_vals[best_r2]+0.025, '🏆', ha='center', fontsize=16)

# RMSE comparison
bars_rmse = axes[1].bar(model_names, rmse_vals, color=palette_cmp, alpha=0.88)
bars_rmse[best_rmse].set_edgecolor('gold'); bars_rmse[best_rmse].set_linewidth(3)
axes[1].set_ylabel('RMSE ($)', fontsize=12)
axes[1].set_title('Test RMSE — All Models\n(lower is better)', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=20)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))
for bar, val in zip(bars_rmse, rmse_vals):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                 f'${val/1000:.1f}K', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].text(best_rmse, rmse_vals[best_rmse]+2000, '🏆', ha='center', fontsize=16)

plt.suptitle('Final Model Comparison Dashboard', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## ✅ Summary — What We Covered

| Step | Topic | Key Takeaway |
|------|-------|--------------|
| 1  | Dataset Loading | 1,460 houses, 20 features, target = SalePrice |
| 2  | EDA | GrLivArea, OverallQual strongest correlations |
| 3  | Preprocessing | Imputed 30 NaN values, removed outliers |
| 4  | Feature Engineering | Created TotalSF, HouseAge, QualArea interaction |
| 5  | Assumption Checks | LINNE verified via plots & VIF |
| 6  | Simple LR | Y = b0 + b1×GrLivArea |
| 7  | Multiple LR | 16 features, best base model |
| 8  | Metrics | R², Adj-R², RMSE, MAE, MAPE |
| 9  | Residual Analysis | Q-Q plot, Shapiro-Wilk, Scale-Location |
| 10 | Ridge | L2 penalty, all features kept |
| 11 | Lasso | L1 penalty, feature selection |
| 12 | ElasticNet | Combined penalty, best for correlated features |
| 13 | Cross-Validation | 5-Fold CV, no data leakage |
| 14 | Polynomial LR | Degrees 1, 2, 3 comparison |
| 15 | Feature Importance | Coefficients + Lasso selection |
| 16 | Learning Curve | Bias-variance diagnosis |
| 17 | Real Prediction | 5 new house profiles predicted |
| 18 | Model Comparison | Ridge/Lasso outperform Simple LR |

---

**Notebook by: Sruthi Tarimana**  
*Complete Linear Regression reference — Kaggle House Prices structure*


In [ ]:
print("=" * 60)
print("  ✅ LINEAR REGRESSION NOTEBOOK COMPLETE")
print("  📌 Author: Sruthi Tarimana")
print("  📊 Dataset: House Prices (Kaggle-style)")
print("  🔢 Models: Simple LR, Multiple LR, Ridge,")
print("             Lasso, ElasticNet, Polynomial LR")
print("  📏 Metrics: R², Adj-R², RMSE, MAE, MAPE")
print("  📐 Checks: LINNE assumptions, VIF, residuals")
print("=" * 60)
